# Q2 风险分位数策略最终版

最终预测模型已经冻结：

$$
W=60,\quad K_L=3,\quad K_P=2,
$$

$$
h_L=lag1,\qquad h_P=lag1+lag7+mean7,
$$

$$
C_L=weekday7,\qquad C_P=annual1,
$$

$$
\alpha_L=0.3,\qquad \alpha_P=10.
$$

最近 28 天成对残差形成净负荷经验场景。

风险分位数只在 **2025-05-01 至 2025-08-31 开发期**扫描

$$
q\in\{0.05,0.10,\ldots,0.95\},
$$

以实际总费用最小为目标确定全局实施参数 $q^*$。Sep–Dec 的预测评价不参与 $q^*$ 选择。


In [2]:
from pathlib import Path
import sys, pandas as pd
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
from q2_final import run_all

report = run_all(
    ROOT/"input"/"q2_actual_load_pv.csv",
    ROOT/"input"/"q2_fixed_prices.csv",
    ROOT/"rerun_results",
)
report


{'q_star': 0.85,
 'development_period': ['2025-05-01', '2025-08-31'],
 'evaluation_period': ['2025-02-01', '2025-12-31'],
 'evaluation_days': 334,
 'planned_purchase_kwh': 22076485.272805803,
 'emergency_purchase_kwh': 225320.99023033743,
 'planned_cost_yuan': 13456454.710735612,
 'emergency_cost_yuan': 876478.6500762139,
 'total_cost_yuan': 14332933.360811826,
 'status': 'PASS'}

In [3]:
scan = pd.read_csv(ROOT/"rerun_results"/"qstar_development_scan.csv")
display(scan.sort_values("total_cost_yuan").round(4))


,q,planned_cost_yuan,emergency_cost_yuan,total_cost_yuan,emergency_kwh,unused_kwh,emergency_days,delta_cost_yuan,delta_pct
16,0.85,4.996425e+06,4.292878e+05,5.425712e+06,1.085066e+05,1.301255e+06,123,0.000000e+00,0.0000
17,0.90,5.128651e+06,3.023386e+05,5.430989e+06,7.656297e+04,1.463737e+06,123,5.276784e+03,0.0973
15,0.80,4.893503e+06,5.636966e+05,5.457200e+06,1.425520e+05,1.179914e+06,123,3.148707e+04,0.5803
14,0.75,4.803981e+06,7.055354e+05,5.509517e+06,1.782111e+05,1.080683e+06,123,8.380439e+04,1.5446
18,0.95,5.322900e+06,1.868619e+05,5.509762e+06,4.758189e+04,1.716596e+06,123,8.404927e+04,1.5491
13,0.70,4.727457e+06,8.524635e+05,5.579921e+06,2.152372e+05,1.001938e+06,123,1.542086e+05,2.8422
12,0.65,4.659317e+06,1.007198e+06,5.666515e+06,2.543866e+05,9.377549e+05,123,2.408027e+05,4.4382
11,0.60,4.594832e+06,1.176189e+06,5.771021e+06,2.972603e+05,8.816596e+05,123,3.453086e+05,6.3643
10,0.55,4.531682e+06,1.359836e+06,5.891518e+06,3.437079e+05,8.312648e+05,123,4.658051e+05,8.5851
9,0.50,4.470771e+06,1.556576e+06,6.027347e+06,3.941439e+05,7.870938e+05,123,6.016349e+05,11.0886


开发期经验最优为

$$
\boxed{q^*=0.85}.
$$

单时段五倍应急损失给出的理论参考为

$$
q_0=1-\frac{1}{5}=0.80.
$$

$q^*=0.85$ 略高于单时段理论值，反映了储能的跨时段耦合、效率损失与分时电价共同作用。


In [4]:
daily = pd.read_csv(ROOT/"rerun_results"/"final_strategy_daily.csv")
daily[["planned_grid_kwh","emergency_kwh","planned_cost_yuan","emergency_cost_yuan","total_cost_yuan"]].sum()


planned_grid_kwh       2.207649e+07
emergency_kwh          2.253210e+05
planned_cost_yuan      1.345645e+07
emergency_cost_yuan    8.764787e+05
total_cost_yuan        1.433293e+07
dtype: float64

## 因果口径

每日预测特征和残差场景只使用决策日之前已经结束的真实日数据。

$q^*$ 以及预测超参数属于**离线全局校准参数**，固定后用于完整策略回放；它们不会在某一天的在线决策中读取当天未来真实负荷或光伏。
